# BaristaBot: A LangGraph + Gemini Cafe Ordering Agent
Author: arielzin33@gmail.com

A stateful conversational agent, built with LangGraph and the Gemini API (via `langchain-google-genai`), that takes coffee/tea orders using real tool calls (menu lookup, add-to-order, remove-from-order, view-order, confirm-order), loops with the user until an order is confirmed, and routes tool calls through LangGraph's `ToolNode` + `tools_condition` mechanism.

**Verified before writing this notebook:** the core graph pattern below (`StateGraph` + `add_messages` + `ToolNode` + `tools_condition`, looping `tools → model` until the model stops calling tools) was tested end-to-end against `langgraph==1.2.11` with a stand-in model that issued a sequence of tool calls — confirmed the graph correctly executes multiple chained tool calls within a single turn and routes back to `END` once the model responds with plain text. Swapping in the real `ChatGoogleGenerativeAI` model below only requires a valid Gemini API key; the graph wiring itself does not change.

---
## Part 1: Setup

In [ ]:
!pip install -q langgraph langchain-google-genai langchain-core


In [ ]:
import os
import getpass

# Get a free Gemini API key at https://aistudio.google.com/app/apikey
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")


---
## Part 2: Define State and Tools

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages


class BaristaState(TypedDict):
    """Conversation state: the running message history plus a live order flag."""
    messages: Annotated[list, add_messages]
    order_confirmed: bool


### Tools

Tools are plain Python functions decorated with `@tool` — LangGraph's `ToolNode` inspects the model's `tool_calls` and executes the matching function automatically. The order itself is kept in a small module-level dict (`ORDER`) rather than inside the LangGraph state, since tool functions in this simple pattern don't have direct write access to the graph state — this keeps the example focused on the graph/routing mechanics that the assignment asks for.

In [ ]:
from langchain_core.tools import tool

MENU = {
    "Espresso": 3.00,
    "Americano": 3.50,
    "Latte": 4.50,
    "Cappuccino": 4.00,
    "Mocha": 5.00,
    "Green Tea": 3.50,
    "Chai Latte": 4.50,
}

ORDER: list[str] = []


@tool
def get_menu() -> str:
    """Returns the full cafe menu with prices, so the assistant can tell the customer what's available."""
    lines = [f"- {name}: ${price:.2f}" for name, price in MENU.items()]
    return "Today's menu:\n" + "\n".join(lines)


@tool
def add_to_order(item: str) -> str:
    """Adds one drink to the customer's current order. `item` must exactly match a menu item name."""
    if item not in MENU:
        closest = ", ".join(MENU.keys())
        return f"'{item}' is not on the menu. Available items: {closest}"
    ORDER.append(item)
    return f"Added {item} to the order. Current order: {ORDER}"


@tool
def remove_from_order(item: str) -> str:
    """Removes one instance of a drink from the customer's current order, if present."""
    if item in ORDER:
        ORDER.remove(item)
        return f"Removed {item}. Current order: {ORDER}"
    return f"{item} is not currently in the order. Current order: {ORDER}"


@tool
def view_order() -> str:
    """Returns the current (unconfirmed) order and running total."""
    if not ORDER:
        return "The order is currently empty."
    total = sum(MENU[i] for i in ORDER)
    return f"Current order: {ORDER}, total: ${total:.2f}"


@tool
def confirm_order() -> str:
    """Finalizes the order. Call this only after the customer explicitly confirms they're done."""
    if not ORDER:
        return "Cannot confirm an empty order — ask the customer what they'd like first."
    total = sum(MENU[i] for i in ORDER)
    return f"ORDER_CONFIRMED: {ORDER}, total: ${total:.2f}. Thank you!"


tools = [get_menu, add_to_order, remove_from_order, view_order, confirm_order]


---
## Part 3: Build the LangGraph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import SystemMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI

SYSTEM_PROMPT = SystemMessage(content=(
    "You are BaristaBot, a friendly cafe assistant. Help the customer browse the menu, build "
    "an order, and confirm it when they're ready. Always use the provided tools to check the "
    "menu, add/remove items, or confirm the order — never invent menu items or prices yourself. "
    "Ask clarifying questions if the customer's request is ambiguous. Keep responses short and "
    "warm."
))

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3)
llm_with_tools = llm.bind_tools(tools)


def chatbot_node(state: BaristaState) -> dict:
    """Calls Gemini with the full message history (system prompt + conversation so far)."""
    messages = [SYSTEM_PROMPT] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


def order_status_node(state: BaristaState) -> dict:
    """Checks the most recent tool result for the ORDER_CONFIRMED marker and updates state."""
    for m in reversed(state["messages"]):
        if isinstance(m, ToolMessage) and "ORDER_CONFIRMED" in str(m.content):
            return {"order_confirmed": True}
        if isinstance(m, ToolMessage):
            break
    return {}


tool_node = ToolNode(tools)

graph = StateGraph(BaristaState)
graph.add_node("chatbot", chatbot_node)
graph.add_node("tools", tool_node)
graph.add_node("order_status", order_status_node)

graph.add_edge(START, "chatbot")
# If the model's last message contains tool_calls, route to "tools"; otherwise route to END.
graph.add_conditional_edges("chatbot", tools_condition, {"tools": "tools", "__end__": END})
graph.add_edge("tools", "order_status")
graph.add_edge("order_status", "chatbot")  # loop back so the model can react to the tool result

app = graph.compile()


### Graph diagram (optional — requires `pip install pygraphviz` or renders as Mermaid)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Diagram rendering needs extra deps in this environment; skipping.", e)
    print(app.get_graph().draw_mermaid())


---
## Part 4: The Conversational Loop

This drives multiple *human* turns — each `app.invoke()` call already resolves an arbitrary chain of tool calls internally (e.g., the model might call `get_menu` then `add_to_order` in the same turn before replying in plain text), and the outer `while` loop here just keeps the conversation going across turns until `order_confirmed` is set.

In [ ]:
from langchain_core.messages import HumanMessage


def run_baristabot():
    state: BaristaState = {"messages": [], "order_confirmed": False}
    print("BaristaBot: Hi! Welcome in \u2014 what can I get started for you today? "
          "(type 'quit' to leave)")

    while not state["order_confirmed"]:
        user_input = input("You: ")
        if user_input.strip().lower() in {"quit", "exit"}:
            print("BaristaBot: No problem, see you next time!")
            break

        state["messages"].append(HumanMessage(content=user_input))
        state = app.invoke(state)

        last_ai_message = state["messages"][-1]
        print("BaristaBot:", last_ai_message.content)

        if state.get("order_confirmed"):
            print("\n[Order confirmed \u2014 conversation loop ends here.]")


# Uncomment to run interactively (needs a real Gemini API key entered above):
# run_baristabot()


### Example scripted run (no `input()` needed — for grading/demo without live typing)

Simulates a realistic customer conversation by feeding a fixed list of user messages instead of reading from `input()`, so the notebook can be run end-to-end non-interactively.

In [ ]:
def run_baristabot_scripted(user_turns: list[str]):
    state: BaristaState = {"messages": [], "order_confirmed": False}
    print("BaristaBot: Hi! Welcome in \u2014 what can I get started for you today?\n")

    for turn in user_turns:
        print("You:", turn)
        state["messages"].append(HumanMessage(content=turn))
        state = app.invoke(state)
        print("BaristaBot:", state["messages"][-1].content, "\n")
        if state.get("order_confirmed"):
            print("[Order confirmed \u2014 loop ends.]")
            break

    return state


demo_turns = [
    "What's on the menu?",
    "I'll take a latte and a cappuccino please.",
    "Actually, remove the cappuccino and add a mocha instead.",
    "What's my order so far?",
    "That's everything, please confirm it.",
]

final_state = run_baristabot_scripted(demo_turns)


---
## Observations & Design Notes

- **`tools_condition`** is LangGraph's built-in router: it inspects the latest `AIMessage` for a non-empty `tool_calls` list and returns `"tools"` if present, `"__end__"` otherwise — this is what implements the "loop until no more tool calls, then respond" behavior without any manual `if/else` routing code.
- **The `order_status` node** is a small custom addition beyond the minimal tool-calling loop: it inspects tool results for the `ORDER_CONFIRMED` marker returned by `confirm_order()` and flips `order_confirmed` in the graph state, which the outer Python loop then reads to decide when to stop prompting the customer for more input — this is the "loop through conversation until an order is placed" requirement from the assignment, implemented as a state field rather than a special exception/return value.
- **Global `ORDER` list vs. graph state:** tool functions in this pattern don't get direct read/write access to `BaristaState` without extra plumbing (`InjectedState`), so the running order is tracked in a simple module-level list instead — sufficient for a single-session demo, but a production version would move order data into the graph state (via `InjectedState`) or an external database so multiple concurrent conversations don't share one order.
- **Why Gemini 2.0 Flash:** chosen for low latency and free-tier availability, which suits an interactive ordering assistant where response speed matters more than maximum reasoning depth; swap `model="gemini-2.0-flash"` for a larger Gemini model if you want stronger disambiguation on more complex, multi-item orders.